# Classes and Object-Oriented Programming in Python

Object-oriented programming (OOP) organizes code around **objects** that combine:

- **data**: the state of an object;
- **behavior**: operations the object can perform.

In this notebook we start from the basic ideas and progressively move toward patterns that appear in real Python and machine-learning code.

## Learning objectives

By the end of this notebook, you should be able to:

- define a class and create objects;
- understand `self`, attributes, methods, and constructors;
- distinguish instance attributes from class attributes;
- use properties for computed or validated attributes;
- understand inheritance, method overriding, and polymorphism;
- understand why composition is often preferable to inheritance;
- use `@classmethod`, `@staticmethod`, and `@dataclass`;
- recognize useful special methods such as `__repr__` and `__len__`;
- understand abstract base classes;
- recognize OOP patterns used by machine-learning libraries;
- define and train a small neural network as a Python class.

# Part I — The basic idea

## 1. From data to objects

A dictionary can represent an entity such as a patient:

In [1]:
patient = {
    "name": "Anna",
    "age": 42,
    "temperature": 37.1
}

print(patient["name"])

Anna


A dictionary stores data, but it does not naturally define the **behavior** associated with that data.

A class lets us define a new type of object.

In [2]:
class Patient:
    pass


p = Patient()

print(p)
print(type(p))

<class '__main__.Patient'>


`Patient` is the **class**.

`p` is an **instance** (an object created from that class).

```text
Class = blueprint
        ↓
Patient
        ↓
   ┌────┴────┐
   ↓         ↓
 Anna       Marco
 objects / instances
```

### Class, object, and instance: terminology

These words are closely related but are not interchangeable.

- A **class** is a definition of a new type.
- An **object** is a value that exists in memory.
- An **instance** is an object created from a particular class.

For example:

```python
class Patient:
    pass

anna = Patient()
```

`Patient` is the class, while `anna` is an object and, more specifically, an instance of `Patient`.

We can check this explicitly:

In [3]:
class Patient:
    pass

anna = Patient()

print(type(anna))
print(isinstance(anna, Patient))

<class '__main__.Patient'>
True


In Python, classes are themselves objects too. This is one of the reasons the language is so flexible.

For example:

```python
type(Patient)
```

returns `type`, because a class is an object whose type is `type`.

In [4]:
print(type(Patient))

<class 'type'>


## 2. The constructor: `__init__`

The method `__init__` is called when a new object is created.

It is commonly used to initialize the object's attributes.

In [5]:
class Patient:
    def __init__(self, name, age, temperature):
        self.name = name
        self.age = age
        self.temperature = temperature


patient = Patient("Anna", 42, 37.1)

print(patient.name)
print(patient.age)
print(patient.temperature)

Anna
42
37.1


### `__init__` initializes an object; it does not create it

A common simplification is to call `__init__` the "constructor".

More precisely:

1. Python creates the object;
2. Python then calls `__init__` to initialize it.

Object creation is handled by another special method, `__new__`, which is rarely customized in ordinary Python code.

For most applications it is perfectly reasonable to think of `__init__` as the constructor, but it is useful to know the distinction.

## 3. What is `self`?

`self` refers to the **current object**.

When we create:

```python
anna = Patient("Anna", 42, 37.1)
marco = Patient("Marco", 67, 38.4)
```

each object has its own values:

```text
anna.name  → "Anna"
marco.name → "Marco"
```

Inside the class, `self.name` means:

> the `name` attribute of this particular object.

In [6]:
anna = Patient("Anna", 42, 37.1)
marco = Patient("Marco", 67, 38.4)

print(anna.name)
print(marco.name)

Anna
Marco


### Method calls and implicit `self`

When we write:

```python
anna.has_fever()
```

Python conceptually performs something similar to:

```python
Patient.has_fever(anna)
```

The instance is automatically passed as the first argument.

This is why instance methods need a first parameter conventionally named `self`.

In [7]:
class Patient:
    def __init__(self, temperature):
        self.temperature = temperature

    def has_fever(self):
        return self.temperature >= 38.0


anna = Patient(38.2)

print(anna.has_fever())
print(Patient.has_fever(anna))

True
True


The two calls above produce the same result.

`self` is not a Python keyword: technically we could use another parameter name. However, using `self` is a very strong convention and should almost always be followed.

## 4. Methods: objects can have behavior

A **method** is a function defined inside a class.

Here the patient object can determine whether it has a fever.

In [8]:
class Patient:
    def __init__(self, name, age, temperature):
        self.name = name
        self.age = age
        self.temperature = temperature

    def has_fever(self, threshold=38.0):
        return self.temperature >= threshold


anna = Patient("Anna", 42, 37.0)
marco = Patient("Marco", 67, 38.4)

print("Anna:", anna.has_fever())
print("Marco:", marco.has_fever())

Anna: False
Marco: True


Notice the call:

```python
marco.has_fever()
```

Conceptually, Python passes `marco` as `self`.

The method can therefore access the state stored inside that object.

## 5. State can change

Objects can maintain state over time.

In [9]:
patient = Patient("Anna", 42, 37.1)

print("Before:", patient.temperature)

patient.temperature = 38.3

print("After:", patient.temperature)
print("Has fever:", patient.has_fever())

Before: 37.1
After: 38.3
Has fever: True


### Object identity and references

Variables in Python hold **references to objects**.

This matters with mutable objects.

If two variables refer to the same object, modifying the object through one variable is visible through the other.

In [10]:
class Patient:
    def __init__(self, name, temperature):
        self.name = name
        self.temperature = temperature


anna = Patient("Anna", 37.1)
same_patient = anna

same_patient.temperature = 38.5

print(anna.temperature)
print(same_patient.temperature)

print(anna is same_patient)

38.5
38.5
True


The operator `is` checks **identity**: whether two references point to the same object.

This is different from `==`, which usually checks **equality of values**.

```text
is  → same object?
==  → equivalent value?
```

For custom classes, `==` behaves like identity unless equality behavior is explicitly defined, for example with `__eq__` or a dataclass.

# Part II — Making classes more useful

## 6. `__repr__`: a useful representation

Special methods surrounded by double underscores are often called **dunder methods**.

`__repr__` controls how an object is represented, which is especially useful for debugging and notebooks.

In [11]:
class Patient:
    def __init__(self, name, age, temperature):
        self.name = name
        self.age = age
        self.temperature = temperature

    def has_fever(self, threshold=38.0):
        return self.temperature >= threshold

    def __repr__(self):
        return (
            f"Patient(name={self.name!r}, "
            f"age={self.age}, "
            f"temperature={self.temperature})"
        )


patient = Patient("Anna", 42, 37.1)
patient

Patient(name='Anna', age=42, temperature=37.1)

### `__repr__` vs `__str__`

Python has two related string representations.

- `__repr__` is mainly intended for developers and debugging.
- `__str__` is mainly intended for a user-friendly representation.

`repr(obj)` uses `__repr__`.

`str(obj)` and `print(obj)` prefer `__str__` if it exists.

In [12]:
class Patient:
    def __init__(self, name, temperature):
        self.name = name
        self.temperature = temperature

    def __repr__(self):
        return f"Patient(name={self.name!r}, temperature={self.temperature!r})"

    def __str__(self):
        return f"{self.name}: {self.temperature:.1f} °C"


patient = Patient("Anna", 37.1)

print("repr:", repr(patient))
print("str :", str(patient))
print("print:", patient)

repr: Patient(name='Anna', temperature=37.1)
str : Anna: 37.1 °C
print: Anna: 37.1 °C


A useful convention is for `__repr__` to be **unambiguous** and informative.

When practical, it can resemble valid Python code that could recreate the object:

```python
Patient(name='Anna', temperature=37.1)
```

The expression `{self.name!r}` explicitly asks for the `repr()` representation of the value.

## 7. Encapsulation

A class groups related **data and behavior** together.

For example, a rectangle stores its dimensions and knows how to compute quantities derived from them.

In [13]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


rectangle = Rectangle(4, 3)

print("Area:", rectangle.area())
print("Perimeter:", rectangle.perimeter())

Area: 12
Perimeter: 14


### Encapsulation does not mean "everything must be private"

In Python, encapsulation usually means:

> expose a clear public interface and keep implementation details internal.

Python relies heavily on conventions rather than strict access restrictions.

Typical naming conventions are:

```text
attribute      public
_attribute     internal-use convention
__attribute    name mangling
```

A single underscore does not prevent access:

In [14]:
class Example:
    def __init__(self):
        self.public = 1
        self._internal = 2


obj = Example()

print(obj.public)
print(obj._internal)   # possible, but conventionally internal

1
2


A double leading underscore triggers **name mangling**:

In [42]:
class Example:
    def __init__(self):
        self.__secret = 42


obj = Example()

print(obj.__dict__)

# This would fail:
#print(obj.__secret)

# But the value is still technically accessible:
print(obj._Example__secret)

{'_Example__secret': 42}
42


Name mangling is mainly designed to reduce accidental name clashes in subclasses. It is **not a security mechanism**.

## 8. Properties

A property lets us expose a method as if it were an attribute.

This is useful for **computed values**.

In [16]:
import math


class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return math.pi * self.radius**2


circle = Circle(2)

print(circle.radius)
print(circle.area)      # no parentheses

2
12.566370614359172


Properties are also useful when we want to **validate assignments**.

Here we prevent a negative temperature in Kelvin.

In [17]:
class Sample:
    def __init__(self, temperature):
        self.temperature = temperature

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        if value < 0:
            raise ValueError("Temperature in Kelvin cannot be negative.")
        self._temperature = value


sample = Sample(300)
print(sample.temperature)

# Try:
# sample.temperature = -10

300


The underscore in `_temperature` is a Python convention meaning:

> this attribute is intended for internal use.

It is not truly private, but users of the class are encouraged to interact through the public interface `temperature`.

### Why properties are useful

Suppose we begin with a simple public attribute:

```python
sample.temperature
```

Later we realize that assignments must be validated.

Without properties, we might need to change the public interface to:

```python
sample.set_temperature(...)
```

A property lets us keep the convenient attribute syntax while adding logic behind it:

```python
sample.temperature = 300
```

The class can therefore evolve internally without forcing users to change how they access the value.

In [18]:
class Sample:
    def __init__(self, temperature):
        self.temperature = temperature

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        value = float(value)

        if value < 0:
            raise ValueError("Kelvin temperature cannot be negative.")

        self._temperature = value


sample = Sample("300")
print(sample.temperature)
print(type(sample.temperature))

300.0
<class 'float'>


## 9. Class attributes vs instance attributes

An **instance attribute** belongs to one object.

A **class attribute** is shared conceptually by all objects of the class.

In [19]:
class Patient:
    fever_threshold = 38.0  # class attribute

    def __init__(self, name, temperature):
        self.name = name              # instance attribute
        self.temperature = temperature

    def has_fever(self):
        return self.temperature >= self.fever_threshold


anna = Patient("Anna", 37.5)
marco = Patient("Marco", 38.3)

print(anna.has_fever())
print(marco.has_fever())
print(Patient.fever_threshold)

False
True
38.0


### A subtle trap: mutable class attributes

Class attributes are shared through the class.

That is convenient for constants such as:

```python
fever_threshold = 38.0
```

but it can be dangerous with mutable objects such as lists.

In [20]:
class Patient:
    notes = []   # shared mutable class attribute

    def __init__(self, name):
        self.name = name


anna = Patient("Anna")
marco = Patient("Marco")

anna.notes.append("Follow-up required")

print("Anna :", anna.notes)
print("Marco:", marco.notes)

Anna : ['Follow-up required']
Marco: ['Follow-up required']


Both instances see the same list because `notes` belongs to the class.

Usually, per-object mutable state should be created inside `__init__`:

In [21]:
class Patient:
    def __init__(self, name):
        self.name = name
        self.notes = []


anna = Patient("Anna")
marco = Patient("Marco")

anna.notes.append("Follow-up required")

print("Anna :", anna.notes)
print("Marco:", marco.notes)

Anna : ['Follow-up required']
Marco: []


This is conceptually similar to the **mutable default argument** pitfall: the problem is unintended sharing of one mutable object.

# Part III — More advanced class features

## 10. `@classmethod`

A class method receives the **class** itself (`cls`) rather than an individual object.

A common use is to define an **alternative constructor**.

In [22]:
class Patient:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year, current_year=2026):
        age = current_year - birth_year
        return cls(name, age)


anna = Patient.from_birth_year("Anna", 1984)

print(anna.name)
print(anna.age)

Anna
42


### Why use `cls` instead of the class name?

Inside a class method, prefer:

```python
return cls(...)
```

rather than:

```python
return Patient(...)
```

Using `cls` preserves inheritance: if a subclass calls the class method, the method can create an instance of that subclass rather than always constructing the parent class.

In [23]:
class Patient:
    def __init__(self, name):
        self.name = name

    @classmethod
    def anonymous(cls):
        return cls("Unknown")


class ResearchPatient(Patient):
    pass


p = ResearchPatient.anonymous()

print(type(p))

<class '__main__.ResearchPatient'>


## 11. `@staticmethod`

A static method belongs conceptually to the class, but it does not need either `self` or `cls`.

It behaves like a normal function grouped inside the class namespace.

In [24]:
class Temperature:
    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return celsius * 9 / 5 + 32


print(Temperature.celsius_to_fahrenheit(37))

98.6


A useful mental model:

```text
instance method   → needs an object        → self
class method      → needs the class        → cls
static method     → needs neither
```

### Do not create a class only to hold static methods

A static method is useful when an operation belongs conceptually to a class namespace.

However, Python already has ordinary functions and modules.

If a class contains only unrelated static methods and no meaningful state, a module with normal functions may be simpler.

For example:

```python
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32
```

may be clearer than creating a `TemperatureUtilities` class solely to group functions.

Use classes when the abstraction is genuinely useful, not simply because OOP is available.

## 12. Dataclasses

Many classes mainly store data.

Python's `@dataclass` decorator can automatically generate methods such as:

- `__init__`
- `__repr__`
- `__eq__`

This removes repetitive boilerplate.

In [25]:
from dataclasses import dataclass


@dataclass
class PatientRecord:
    name: str
    age: int
    temperature: float


anna = PatientRecord("Anna", 42, 37.1)
marco = PatientRecord("Marco", 67, 38.4)

print(anna)
print(anna == marco)

PatientRecord(name='Anna', age=42, temperature=37.1)
False


Type annotations such as:

```python
name: str
age: int
```

document the intended types.

Python does **not** automatically enforce them at runtime, but they help readers, IDEs, and static type checkers.

### Dataclass defaults and `field(default_factory=...)`

Dataclasses have the same fundamental issue with shared mutable defaults.

For per-instance lists, use `default_factory`.

In [26]:
from dataclasses import dataclass, field


@dataclass
class PatientRecord:
    name: str
    notes: list[str] = field(default_factory=list)


anna = PatientRecord("Anna")
marco = PatientRecord("Marco")

anna.notes.append("Follow-up")

print(anna)
print(marco)

PatientRecord(name='Anna', notes=['Follow-up'])
PatientRecord(name='Marco', notes=[])


`default_factory=list` tells the dataclass to create a **new list for every instance**.

Dataclasses can also be made immutable-like using `frozen=True`:

In [27]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Point:
    x: float
    y: float


p = Point(1.0, 2.0)

print(p)

# This would raise FrozenInstanceError:
# p.x = 10

Point(x=1.0, y=2.0)


`frozen=True` is useful for value-like objects that should not change after creation.

It is not absolute deep immutability: a frozen dataclass can still contain mutable objects internally.

## 13. Other useful dunder methods

Classes can behave like built-in Python objects by implementing special methods.

For example, `__len__` lets us use `len(object)`.

In [28]:
class MeasurementSeries:
    def __init__(self, values):
        self.values = list(values)

    def __len__(self):
        return len(self.values)

    def __getitem__(self, index):
        return self.values[index]


series = MeasurementSeries([10, 12, 15, 9])

print(len(series))
print(series[0])
print(series[1:3])

4
10
[12, 15]


Because we implemented `__len__` and `__getitem__`, our custom object starts to behave like a Python container.

### Iteration with `__iter__`

We can make a custom object directly iterable by implementing `__iter__`.

In [29]:
class MeasurementSeries:
    def __init__(self, values):
        self.values = list(values)

    def __len__(self):
        return len(self.values)

    def __getitem__(self, index):
        return self.values[index]

    def __iter__(self):
        return iter(self.values)


series = MeasurementSeries([10, 12, 15, 9])

for value in series:
    print(value)

10
12
15
9


Many Python operations are based on **protocols** rather than on a specific class hierarchy.

For example, an object can participate in iteration if it provides the expected iteration behavior.

Other common dunder methods include:

```text
__contains__   → x in object
__eq__         → object1 == object2
__lt__         → object1 < object2
__add__        → object1 + object2
__call__       → object(...)
```

Implement dunder methods only when their meaning is natural for the abstraction.

In [30]:
class MeasurementSeries:
    def __init__(self, values):
        self.values = list(values)

    def __contains__(self, value):
        return value in self.values


series = MeasurementSeries([10, 12, 15, 9])

print(12 in series)
print(100 in series)

True
False


# Part IV — Relationships between classes

## 14. Inheritance

Inheritance lets one class reuse and extend another class.

The child class inherits attributes and methods from the parent class.

In [31]:
class Person:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"Person: {self.name}"


class Student(Person):
    def __init__(self, name, student_id):
        super().__init__(name)
        self.student_id = student_id

    def describe(self):
        return f"Student: {self.name}, ID={self.student_id}"


student = Student("Elena", "S001")

print(student.name)
print(student.describe())

Elena
Student: Elena, ID=S001


`super()` lets the child class reuse behavior from its parent.

`Student.describe()` **overrides** `Person.describe()`.

## 15. Polymorphism

Different classes can provide the same interface.

The calling code does not necessarily need to know the exact class.

In [32]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius**2


class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


shapes = [
    Circle(2),
    Rectangle(3, 4),
    Circle(1)
]

for shape in shapes:
    print(type(shape).__name__, "area =", shape.area())

Circle area = 12.566370614359172
Rectangle area = 12
Circle area = 3.141592653589793


The loop only assumes:

```python
shape.area()
```

It does not care whether `shape` is a `Circle` or a `Rectangle`.

This is **polymorphism**.

### Duck typing

Python often uses **duck typing**:

> If an object provides the behavior we need, its exact class may not matter.

The famous idea is:

> "If it walks like a duck and quacks like a duck, treat it like a duck."

In the previous example, the loop does not ask:

```python
isinstance(shape, Circle)
```

It simply calls:

```python
shape.area()
```

Any object providing a compatible `area()` method can participate.

In [33]:
class Triangle:
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return self.base * self.height / 2


shapes = [
    Circle(2),
    Rectangle(3, 4),
    Triangle(3, 5),
]

for shape in shapes:
    print(f"{type(shape).__name__:10s} area = {shape.area():.2f}")

Circle     area = 12.57
Rectangle  area = 12.00
Triangle   area = 7.50


This form of polymorphism does **not require inheritance**.

That is an important characteristic of Python's object model.

## 16. Abstract base classes

Sometimes we want to explicitly define an interface that subclasses must implement.

Python provides **abstract base classes** through the `abc` module.

In [34]:
from abc import ABC, abstractmethod


class Shape(ABC):

    @abstractmethod
    def area(self):
        pass


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


rectangle = Rectangle(4, 3)
print(rectangle.area())

# Shape() would fail because Shape is abstract.

12


This becomes useful in larger software projects because it makes the expected interface explicit.

### Abstract base classes vs duck typing

Both approaches can define a common interface, but they emphasize different ideas.

**Duck typing**

- flexible;
- no shared parent class required;
- relies on objects supporting the expected operations.

**Abstract base classes**

- make the intended interface explicit;
- can prevent incomplete subclasses from being instantiated;
- useful when designing a controlled family of related classes.

Python code often uses both styles depending on the problem.

## 17. Composition

Inheritance models an **is-a** relationship:

```text
Student is a Person
```

Composition models a **has-a** relationship:

```text
Person has an Address
```

Often, composition is more flexible.

In [35]:
@dataclass
class Address:
    city: str
    country: str


class Person:
    def __init__(self, name, address):
        self.name = name
        self.address = address


address = Address("Trieste", "Italy")
person = Person("Anna", address)

print(person.name)
print(person.address.city)

Anna
Trieste


### Composition and dependency injection

Composition becomes especially powerful when an object receives another object that provides some service.

For example, a patient report might depend on a formatter:

In [36]:
class PlainTextFormatter:
    def format(self, name, temperature):
        return f"{name}: {temperature:.1f} °C"


class PatientReport:
    def __init__(self, formatter):
        self.formatter = formatter

    def create(self, name, temperature):
        return self.formatter.format(name, temperature)


formatter = PlainTextFormatter()
report = PatientReport(formatter)

print(report.create("Anna", 37.2))

Anna: 37.2 °C


`PatientReport` does not create the formatter itself. The formatter is **injected** from outside.

This has useful consequences:

- the report class is less tightly coupled to one formatting strategy;
- the formatter can be replaced;
- testing becomes easier.

For example:

In [37]:
class UpperCaseFormatter:
    def format(self, name, temperature):
        return f"{name}: {temperature:.1f} °C".upper()


report = PatientReport(UpperCaseFormatter())

print(report.create("Anna", 37.2))

ANNA: 37.2 °C


This is a simple example of **dependency injection**: provide an object's dependencies from outside rather than hard-coding them internally.

## Inheritance or composition?

A useful rule of thumb:

```text
"is a"  → inheritance may make sense
"has a" → composition is usually natural
```

Do not use inheritance only to reuse a few lines of code.

### Multiple inheritance and method resolution order

Python supports multiple inheritance:

```python
class C(A, B):
    ...
```

When several parent classes define the same method, Python needs a precise rule for deciding which implementation is used.

That order is called the **Method Resolution Order (MRO)**.

In [38]:
class A:
    def describe(self):
        return "A"


class B:
    def describe(self):
        return "B"


class C(A, B):
    pass


obj = C()

print(obj.describe())
print(C.mro())

A
[<class '__main__.C'>, <class '__main__.A'>, <class '__main__.B'>, <class 'object'>]


Python searches according to the MRO, so `A.describe()` is found before `B.describe()` here.

Multiple inheritance can be useful, especially with small **mixin** classes, but complicated inheritance hierarchies can become difficult to reason about.

For introductory design, favor simple inheritance and composition unless multiple inheritance solves a clear problem.

# Part V — OOP in machine learning

Modern ML libraries make extensive use of classes.

For example, in PyTorch:

```python
class MyNetwork(nn.Module):
    ...
```

A neural network is naturally represented as an object because it has:

- **state**: its trainable parameters;
- **structure**: its layers;
- **behavior**: the forward computation;
- methods inherited from a framework class such as `.parameters()`, `.train()`, and `.eval()`.

## 18. A neural network as a class

We will build a small neural network for a synthetic binary-classification problem.

If PyTorch is not installed in the active environment, install it before running this section.

```bash
python -m pip install torch matplotlib
```

In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

ModuleNotFoundError: No module named 'torch'

### Create a simple dataset

Each observation has two features.

The label is `1` when the point is approximately outside a circle and `0` otherwise. This creates a **non-linear classification problem**.

In [ ]:
n_samples = 1000

X = torch.rand(n_samples, 2) * 2 - 1

distance_squared = X[:, 0]**2 + X[:, 1]**2
y = (distance_squared > 0.5).float().reshape(-1, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(
    X[:, 0].numpy(),
    X[:, 1].numpy(),
    c=y[:, 0].numpy(),
    s=15
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic classification problem")
plt.show()

### Define the neural network

In PyTorch, neural networks inherit from `nn.Module`.

The constructor defines the layers.

The `forward()` method defines how data moves through the network.

In [ ]:
class SimpleClassifier(nn.Module):

    def __init__(self, input_dim=2, hidden_dim=16):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.network(x)


model = SimpleClassifier()

print(model)

This is a good example of several OOP ideas working together:

```text
SimpleClassifier
      │
      └── inherits from nn.Module
                 │
                 ├── parameters()
                 ├── train()
                 ├── eval()
                 └── many other methods

SimpleClassifier also contains other objects:

nn.Linear
nn.ReLU
nn.Sequential
```

So PyTorch uses both **inheritance** and **composition**.

### Calling a PyTorch model: `__call__` behind the scenes

When we write:

```python
logits = model(X)
```

we are calling the model object as if it were a function.

This works because `nn.Module` implements `__call__`.

Its `__call__` machinery eventually invokes our:

```python
forward(self, x)
```

method, while also handling framework functionality such as hooks.

Therefore, in PyTorch we normally write:

```python
model(X)
```

rather than calling:

```python
model.forward(X)
```

directly.

### Inspect the model state

The weights and biases are stored inside the object.

In [ ]:
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))

### Train the model

The optimizer receives the parameters belonging to our model object.

`BCEWithLogitsLoss` combines a sigmoid transformation with binary cross-entropy in a numerically stable implementation.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

loss_history = []

for epoch in range(300):

    model.train()

    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | loss = {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss")
plt.show()

### Evaluate the model

During evaluation we use:

```python
model.eval()
```

and disable gradient computation with:

```python
torch.no_grad()
```

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).float()

accuracy = (predictions == y).float().mean()

print(f"Training accuracy: {accuracy.item():.3f}")

### Visualize the learned decision boundary

In [ ]:
grid_values = torch.linspace(-1, 1, 200)

xx, yy = torch.meshgrid(
    grid_values,
    grid_values,
    indexing="xy"
)

grid = torch.stack(
    [xx.reshape(-1), yy.reshape(-1)],
    dim=1
)

model.eval()

with torch.no_grad():
    grid_prob = torch.sigmoid(model(grid))

zz = grid_prob.reshape(xx.shape)

plt.figure(figsize=(6, 5))

plt.contourf(
    xx.numpy(),
    yy.numpy(),
    zz.numpy(),
    levels=20,
    alpha=0.6
)

plt.scatter(
    X[:, 0].numpy(),
    X[:, 1].numpy(),
    c=y[:, 0].numpy(),
    s=10
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Decision surface learned by the neural network")
plt.show()

## Why is OOP useful for a neural network?

Imagine defining a model only with unrelated variables and functions:

```text
weights_layer_1
bias_layer_1
weights_layer_2
bias_layer_2
forward_function(...)
...
```

As the model grows, it becomes difficult to manage.

Instead, the class gives us one coherent object:

```python
model = SimpleClassifier()
```

that contains:

```text
model
├── architecture
├── weights
├── biases
├── forward computation
├── train/eval mode
└── inherited framework functionality
```

This is one of the reasons OOP is so common in machine learning libraries.

## A small realistic example: separating responsibilities

A useful OOP design often combines several of the ideas above.

Suppose we want to represent a series of clinical measurements and classify its mean value.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class MeasurementSeries:
    patient_id: str
    values: list[float] = field(default_factory=list)

    def add(self, value: float) -> None:
        self.values.append(float(value))

    @property
    def mean(self) -> float | None:
        if not self.values:
            return None
        return sum(self.values) / len(self.values)

    def __len__(self) -> int:
        return len(self.values)


class ThresholdClassifier:
    def __init__(self, threshold: float):
        self.threshold = threshold

    def classify(self, value: float) -> str:
        return "high" if value >= self.threshold else "normal"


series = MeasurementSeries("P001")
series.add(36.8)
series.add(37.1)
series.add(37.4)

classifier = ThresholdClassifier(37.0)

print("Number of measurements:", len(series))
print("Mean:", series.mean)

if series.mean is not None:
    print("Classification:", classifier.classify(series.mean))

Notice the separation of responsibilities:

```text
MeasurementSeries
    → stores and summarizes measurements

ThresholdClassifier
    → applies a classification rule
```

We could have placed everything in one giant class, but smaller classes with clear responsibilities are usually easier to understand and test.

# Part VI — Exercises

## Exercise 1 — Measurement series

Create a class `MeasurementSeries` with:

- an attribute `values`;
- a method `mean()`;
- a method `minimum()`;
- a method `maximum()`;
- a property `count`;
- a `__repr__` method;
- a `__len__` method.

Then create an object and test all methods.

In [ ]:
# Write your solution here

## Possible solution

In [ ]:
class MeasurementSeries:
    def __init__(self, values):
        self.values = list(values)

    def mean(self):
        return sum(self.values) / len(self.values)

    def minimum(self):
        return min(self.values)

    def maximum(self):
        return max(self.values)

    @property
    def count(self):
        return len(self.values)

    def __len__(self):
        return len(self.values)

    def __repr__(self):
        return f"MeasurementSeries(n={len(self)}, values={self.values})"


series = MeasurementSeries([10, 12, 15, 9])

print(series)
print("Mean:", series.mean())
print("Minimum:", series.minimum())
print("Maximum:", series.maximum())
print("Count:", series.count)
print("len(series):", len(series))

## Exercise 2 — Extend the neural network

Modify `SimpleClassifier` so that:

1. the number of hidden layers can be changed;
2. the hidden dimension can be changed;
3. dropout can optionally be added.

Think about which information should become constructor arguments and which objects should be stored as attributes.

## Exercise 3 — Equality and representation

Create a class `Medication` with:

- `name`;
- `dose_mg`;
- a useful `__repr__`;
- an `__eq__` method so that two medications are considered equal when both attributes are equal.

Test:

```python
m1 = Medication("Drug A", 10)
m2 = Medication("Drug A", 10)
m3 = Medication("Drug A", 20)
```

What should `m1 == m2` and `m1 == m3` return?

In [ ]:
# Write your solution here

## Exercise 4 — Composition

Create:

1. a `Sensor` class with a `read()` method;
2. a `Monitor` class that receives a sensor in its constructor;
3. a `Monitor.check()` method that reads the sensor and checks whether the value exceeds a threshold.

Then replace the real sensor with a simple `FakeSensor` returning a fixed value.

This illustrates why composition and dependency injection make code easier to test.

In [ ]:
# Write your solution here

# When should you use classes?

Classes are especially useful when:

- you have many entities with the same structure;
- data and behavior naturally belong together;
- objects maintain state over time;
- several components interact through a common interface;
- you are building reusable software components.

Classes are **not automatically better than functions**.

For a simple transformation such as:

```python
celsius_to_fahrenheit(x)
```

a normal function may be clearer than defining a class.

# Take-home messages

- A **class** defines a new type of object.
- An **object** combines state and behavior.
- `__init__` initializes object state.
- `self` refers to the current instance.
- Methods operate on object state.
- Properties provide controlled or computed attribute access.
- Class methods operate on the class; static methods need neither an instance nor the class.
- `@dataclass` reduces boilerplate for data-oriented classes.
- Dunder methods let custom classes integrate naturally with Python.
- Inheritance expresses an **is-a** relationship.
- Composition expresses a **has-a** relationship and is often more flexible.
- Polymorphism lets different objects expose a common interface.
- Abstract base classes can make an expected interface explicit.
- ML frameworks such as PyTorch use OOP extensively: a neural network object contains its layers, parameters, state, and behavior.

# Advanced concepts recap

A few ideas from this notebook are worth keeping in mind when reading real Python code:

### 1. Variables refer to objects

```text
variable ──► object
```

Two variables can refer to the same mutable object, so changes may be visible through both references.

### 2. Python favors protocols and behavior

An object often does not need to inherit from a particular class. It only needs to support the operations expected by the calling code.

This is the basis of duck typing.

### 3. Inheritance is only one form of reuse

Composition is often more flexible because components can be replaced independently.

### 4. Dunder methods integrate custom classes into Python syntax

```text
len(x)     → x.__len__()
x[i]       → x.__getitem__(i)
x == y     → x.__eq__(y)
x in y     → y.__contains__(x)
x(...)     → x.__call__(...)
```

Normally we use the public syntax (`len(x)`, `x[i]`, etc.) rather than calling dunder methods directly.

### 5. Good OOP is about modeling responsibilities

The goal is not to put everything inside classes.

Use a class when state and behavior naturally belong together, and prefer functions when a simple stateless transformation is enough.